In [1]:
import torch
from pathlib import Path
import os
import json
import pandas as pd
from tqdm import tqdm
from model import DiagnosticModel

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
results_dir = Path('../outputs')
device = 'cuda'

In [4]:
results_dict = {}

for sbdir in tqdm(os.listdir(results_dir )):
    sbdir_path = results_dir / sbdir
    
    model_path = sbdir_path / 'best_model.pth'
    params_path = sbdir_path / 'params.json'
    results_path = sbdir_path / 'test_results.json'

    with open(params_path, 'r') as f:
        params = json.load(f)
    with open(results_path, 'r') as f:
        results = json.load(f)

    checkpoint = torch.load(model_path, map_location=device)
    # model = DiagnosticModel(model_name = params['model'], in_channels = 1)
    # model = model.to(device)
    # model.load_state_dict(checkpoint['model_state_dict'])
    
    start_epoch = checkpoint['epoch']
    best_val_acc = checkpoint['val_acc']
    full_train = checkpoint['full_train']
    full_val = checkpoint['full_val']
    full_loss = checkpoint['full_loss']

    auc = results['auc']
    
    results_dict[sbdir] = (best_val_acc, auc)


100%|██████████| 6/6 [00:17<00:00,  2.93s/it]


In [9]:
sort_metric = 'auc'

df = pd.DataFrame(
    [(k, float(v1), float(v2)) for k, (v1, v2) in results_dict.items()],
    columns=["exp", "val_acc", "auc"]
)

df = df.sort_values(by=sort_metric, ascending=False)
df = df.reset_index(drop=True)
df


,exp,val_acc,auc
0,balance_w_download,0.775670,0.810809
1,balance_b_download,0.798549,0.741229
2,balance_o_download,0.795201,0.739281
3,balance_w,0.522879,0.683031
4,balance_b,0.620536,0.642195
5,balance_o,0.403460,0.584435
